# Hito 2: ETL y Calidad de Datos
## Amazon Sales Report — Ventas de Moda en India

> **Objetivo:** Transformar ~129.000 órdenes de venta en un dataset limpio, tipado y enriquecido, listo para responder las preguntas de negocio del proyecto.

---

### Contexto del Negocio

El dataset registra pedidos de una tienda de **indumentaria femenina** en Amazon India (Sets, Kurtas, Western Dress, Tops, entre otras categorías). Cada fila es una orden de compra con información sobre el estado del envío, el monto cobrado en rupias indias (INR), la ciudad de destino y el canal de venta.

Antes de cualquier análisis, los datos pasan por cuatro etapas obligatorias:

| Etapa | Acción | ¿Por qué importa? |
|-------|--------|-------------------|
| **Auditoría** | Revisar estructura, tipos y valores faltantes | Sin diagnóstico no hay limpieza inteligente |
| **Limpieza** | Tratar nulos, duplicados y outliers | Datos sucios generan conclusiones falsas |
| **Transformación** | Normalizar textos y convertir tipos de dato | Permite agrupar y comparar correctamente |
| **Enriquecimiento** | Crear variables derivadas | Las preguntas de negocio requieren métricas que no existen en el dataset crudo |

## 1. Configuración del Entorno

Importamos las librerías del proyecto y leemos el dataset crudo desde el archivo CSV local.

- **pandas / numpy:** manipulación de datos y cálculos estadísticos
- **matplotlib / seaborn:** visualizaciones (utilizados en los notebooks de análisis posteriores)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv('AmazonSaleReport.csv', low_memory=False)


## 2. Auditoría de Calidad de Datos

Antes de modificar cualquier dato, necesitamos **un diagnóstico completo**. Esta etapa responde cuatro preguntas fundamentales:

1. ¿Cuántos registros y columnas tenemos?
2. ¿Qué tipo de dato almacena cada columna?
3. ¿Cuántos valores le faltan a cada columna?
4. ¿Hay filas duplicadas?

In [2]:
print("Número de filas y columnas (Filas, Columnas):")
print(df_raw.shape)
print("-" * 50)

print("Primeras 5 filas del dataset:")
display(df_raw.head(5)) 

Número de filas y columnas (Filas, Columnas):
(128975, 24)
--------------------------------------------------
Primeras 5 filas del dataset:


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


In [3]:
# Informacion del dataset
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount              121180 non-null  float64


In [4]:
# Conteo exacto de nulos por columna.
print("Conteo de valores nulos por columna:")
df_raw.isnull().sum()

Conteo de valores nulos por columna:


index                     0
Order ID                  0
Date                      0
Status                    0
Fulfilment                0
Sales Channel             0
ship-service-level        0
Style                     0
SKU                       0
Category                  0
Size                      0
ASIN                      0
Courier Status         6872
Qty                       0
currency               7795
Amount                 7795
ship-city                33
ship-state               33
ship-postal-code         33
ship-country             33
promotion-ids         49153
B2B                       0
fulfilled-by          89698
Unnamed: 22           49050
dtype: int64

In [5]:
# 1. Calculas y filtras los mayores a 0
df_percentage = ((df_raw.isnull().sum() / len(df_raw)) * 100)
df_percentage = df_percentage[df_percentage > 0]

# 2. Aplicas el formato con el signo % al final (o al principio si prefieres)
df_percentage_formatted = df_percentage.map("{:.2f}%".format)

print(df_percentage_formatted)

Courier Status       5.33%
currency             6.04%
Amount               6.04%
ship-city            0.03%
ship-state           0.03%
ship-postal-code     0.03%
ship-country         0.03%
promotion-ids       38.11%
fulfilled-by        69.55%
Unnamed: 22         38.03%
dtype: str


In [6]:
# Cantidad de filas duplicadas exactas.
print("Número de filas duplicadas exactas:", df_raw.duplicated().sum())

Número de filas duplicadas exactas: 0


In [7]:
# Estadísticas básicas de columnas numéricas y categóricas.
print("Estadísticas de columnas numéricas:")
df_raw.describe(include='all').T

Estadísticas de columnas numéricas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,128975.0,NaN,NaN,NaN,64487.0,37232.019822,0.0,32243.5,64487.0,96730.5,128974.0
Order ID,128975,120378,403-4984515-8861958,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,128975,91,05-03-22,2085,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,128975,13,Shipped,77804,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fulfilment,128975,2,Amazon,89698,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales Channel,128975,2,Amazon.in,128851,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ship-service-level,128975,2,Expedited,88615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Style,128975,1377,JNE3797,4224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SKU,128975,7195,JNE3797-KR-L,773,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,128975,9,Set,50284,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Hallazgos de la Auditoría

#### Estructura general
El dataset tiene **128.975 registros** y **24 columnas**. Las columnas se distribuyen en:

| Tipo | Cant. | Ejemplos |
|------|-------|---------|
| Texto (`str`) | 18 | `Order ID`, `Status`, `Category`, `ship-city` |
| Decimal (`float64`) | 2 | `Amount`, `ship-postal-code` |
| Entero (`int64`) | 1 | `Qty` |
| Booleano (`bool`) | 1 | `B2B` |
| Objeto sin tipo definido | 1 | `Unnamed: 22` |

---

#### Valores nulos — 10 columnas afectadas

| Columna | Nulos | % del total | Interpretación |
|---------|-------|-------------|----------------|
| `fulfilled-by` | 89.698 | 69,6% | Solo registra envíos gestionados por Amazon; el resto lo gestiona el vendedor directamente |
| `Unnamed: 22` | 49.050 | 38,0% | Columna fantasma sin información útil → **se eliminará** |
| `promotion-ids` | 49.153 | 38,1% | La mayoría de ventas no usa código de descuento |
| `Amount` | 7.795 | 6,0% | Ventas sin monto registrado → requiere **imputación contextual** |
| `currency` | 7.795 | 6,0% | Correlacionado con los nulos en `Amount` |
| `Courier Status` | 6.872 | 5,3% | Pedidos cancelados o sin seguimiento de courier |
| `ship-city / state / postal / country` | 33 c/u | 0,03% | Pedidos con dirección incompleta → **se eliminarán las filas** |

---

#### Duplicados
**No hay filas duplicadas exactas.** No se requiere ninguna acción.

---

#### Estadísticas de `Amount` (monto de venta en INR)

| Métrica | Valor | Lectura de negocio |
|---------|-------|-------------------|
| Mínimo | ₹0 | Cancelaciones o errores de registro |
| Media | ₹648,56 | Ticket promedio saludable |
| Mediana | ₹605 | Distribución levemente asimétrica hacia arriba |
| Máximo | ₹5.584 | Ventas de alto valor — **candidatas a outliers** |
| Desvío estándar | ₹281,21 | Variación moderada alrededor de la media |

#### Observación clave
> **923 registros** tienen `Amount` nulo pero `Courier Status` no nulo: pedidos que se procesaron y enviaron, pero cuyo monto no fue capturado correctamente por el sistema.

## 3. Tratamiento de Valores Nulos

Con el diagnóstico completo, aplicamos un tratamiento **contextual** a cada columna: no imputamos mecánicamente, sino que interpretamos qué significa cada nulo dentro del contexto del negocio.

**Criterios aplicados:**
- **Eliminar filas** → cuando son muy pocas y la información es irrecuperable
- **Rellenar con categoría lógica** → cuando el nulo tiene una interpretación clara de negocio
- **Imputar con mediana por grupo** → cuando el valor faltante es numérico y existe una referencia contextual relevante

In [8]:
# Copia limpia del dataset para trabajar sin modificar el original.
df_clean = df_raw.copy()
print("Número de filas y columnas en el dataset limpio:", df_clean.shape)

Número de filas y columnas en el dataset limpio: (128975, 24)


In [9]:
# Eliminación robusta de columnas redundantes o irrelevantes
columnas_a_eliminar = ['Unnamed: 22', 'index']

df_clean = df_raw.drop(columns=columnas_a_eliminar)
print("Número de filas y columnas después de eliminar columnas irrelevantes:", df_clean.shape)

Número de filas y columnas después de eliminar columnas irrelevantes: (128975, 22)


In [10]:
# Columnas críticas para la logística de envío
columnas_envio = ['ship-city', 'ship-state', 'ship-postal-code', 'ship-country']

# Eliminación de filas con datos de envío incompletos
df_clean = df_clean.dropna(subset=columnas_envio)

print(f"Dimensiones tras limpiar datos de envío: {df_clean.shape}")

Dimensiones tras limpiar datos de envío: (128942, 22)


In [11]:
df_clean['promotion-ids'] = df_clean['promotion-ids'].fillna('Sin promoción') # Rellenar nulos en 'promotion-ids' con 'Sin promoción'.
df_clean['fulfilled-by'] = df_clean['fulfilled-by'].fillna('Vendedor') # Rellenar nulos en 'fulfilled-by' con 'Vendedor'.

# Filtrar filas donde Courier Status es nulo
nulos_courier = df_clean[df_clean['Courier Status'].isnull()]
# Imprimir los valores que toma la columna 'Status' para las filas donde 'Courier Status' es nulo.
print("Filas donde Courier Status es nulo: \n", nulos_courier['Status'].value_counts(), "\n")
'''
Acá observamos que los valores que toma la columna 'Status' para las filas donde 'Courier Status' es nulo son:

1- Cancelled                        (para 6858 filas donde 'Courier Status' es nulo)
2- Shipped - Delivered to Buyer     (para 8 filas donde 'Courier Status' es nulo)
3- Shipped - Returned to Seller     (para 3 filas donde 'Courier Status' es nulo)

Esto nos da una pista de cómo podríamos rellenar los nulos en 'Courier Status' basándonos en el valor de 'Status'. 
'''

# Filtrar filas donde currency es nulo
nulos_currency = df_clean[df_clean['currency'].isnull()]
# Imprimir los valores que toma la columna 'Status' para las filas donde 'currency' es nulo.
print("Filas donde currency es nulo: \n", nulos_currency['Status'].value_counts(), "\n")
'''
Acá observamos que los valores que toma la columna 'Status' para las filas donde 'currency' es nulo son:

1- Cancelled                          (para 7564 filas donde 'currency' es nulo)
2- Shipped                            (para 208 filas donde 'currency' es nulo)
3- Shipped - Delivered to Buyer       (para 8 filas donde 'currency' es nulo)
4- Shipping                           (para 8 filas donde 'currency' es nulo)
5- Shipped - Returned to Seller       (para 3 filas donde 'currency' es nulo)
6- Pending                            (para 2 filas donde 'currency' es nulo)

Esto nos da una pista de cómo podríamos rellenar los nulos en 'currency' basándonos en el valor de 'Status'.
'''

'''
Ejemplo de cómo podríamos rellenar los nulos en 'Courier Status' basándonos en el valor de 'Status' uno por uno:

# Rellenar nulos en 'Courier Status' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Cancelled'), 'Courier Status'] = 'Cancelado' 

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Delivered to Buyer'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Shipped - Delivered to Buyer'), 'Courier Status'] = 'No registrado' 

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Shipped - Returned to Seller'), 'Courier Status'] = 'No registrado'

'''

# Versión usando listas y código más compacto para rellenar nulos en 'Courier Status' basándonos en el valor de 'Status':

# Rellenar nulos en 'Courier Status' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Cancelled'), 'Courier Status'] = 'Cancelado'

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Delivered to Buyer' o 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'].isin(['Shipped - Delivered to Buyer', 'Shipped - Returned to Seller'])), 'Courier Status'] = 'No registrado'

# Verificar que ya no hay nulos en 'Courier Status'
print("¿Quedan nulos en Courier Status?", df_clean['Courier Status'].isna().sum())

'''
Ejemplo de cómo podríamos rellenar los nulos en 'currency' basándonos en el valor de 'Status' uno por uno:

# Rellenar nulos en 'currency' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Cancelled'), 'currency'] = 'Cancelado'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped - Delivered to Buyer'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped - Delivered to Buyer'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipping'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipping'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped - Returned to Seller'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Pending'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Pending'), 'currency'] = 'INR'

'''

# Versión usando listas y código más compacto para rellenar nulos en 'currency' basándonos en el valor de 'Status':

# Rellenar nulos en 'currency' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Cancelled'), 'currency'] = 'Cancelado'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped', 'Shipped - Delivered to Buyer', 'Shipping', 'Shipped - Returned to Seller' o 'Pending'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'].isin(['Shipped', 'Shipped - Delivered to Buyer', 'Shipping', 'Shipped - Returned to Seller', 'Pending'])), 'currency'] = 'INR'

# Verificar que ya no hay nulos
print("¿Quedan nulos en currency?", df_clean['currency'].isna().sum())

Filas donde Courier Status es nulo: 
 Status
Cancelled                       6858
Shipped - Delivered to Buyer       8
Shipped - Returned to Seller       3
Name: count, dtype: int64 

Filas donde currency es nulo: 
 Status
Cancelled                       7564
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
Name: count, dtype: int64 

¿Quedan nulos en Courier Status? 0
¿Quedan nulos en currency? 0


In [12]:
# Primero, asegurarnos de que podemos agrupar por categoría
print("¿Hay nulos en Category?", df_clean['Category'].isna().sum())
print("Categorías disponibles:", df_clean['Category'].unique(), "\n")

# Segundo, calcular el precio mediano para cada categoría de producto
mediana_por_categoria = df_clean.groupby('Category')['Amount'].median()
print("Mediana de Amount por categoría:")
print(mediana_por_categoria, "\n")

# Calcular mediana global (por si acaso)
mediana_global = df_clean['Amount'].median()

# Para cada categoría, rellenar sus nulos con su mediana
for categoria in mediana_por_categoria.index:
    mascara = (df_clean['Category'] == categoria) & (df_clean['Amount'].isna())
    df_clean.loc[mascara, 'Amount'] = mediana_por_categoria[categoria]

# Los nulos que queden (por categoría nula) los rellenamos con mediana global
df_clean['Amount'] = df_clean['Amount'].fillna(mediana_global)

print("Amount limpiado correctamente")
print(f"Última verificación - nulos restantes: {df_clean['Amount'].isna().sum()}")

¿Hay nulos en Category? 0
Categorías disponibles: <ArrowStringArray>
[          'Set',         'kurta', 'Western Dress',           'Top',
  'Ethnic Dress',        'Bottom',         'Saree',        'Blouse',
       'Dupatta']
Length: 9, dtype: str 

Mediana de Amount por categoría:
Category
Blouse           545.00
Bottom           345.00
Dupatta          305.00
Ethnic Dress     837.00
Saree            791.00
Set              788.00
Top              519.05
Western Dress    744.00
kurta            435.00
Name: Amount, dtype: float64 

Amount limpiado correctamente
Última verificación - nulos restantes: 0


### Resumen del Tratamiento de Nulos

| Columna | Estrategia | Valor asignado | Justificación |
|---------|-----------|----------------|---------------|
| `Unnamed: 22` | Eliminar columna | — | Columna sin información útil |
| `ship-city / state / postal / country` | Eliminar 33 filas | — | 0,03% del total; ubicación irrecuperable |
| `promotion-ids` | Rellenar | `"sin promoción"` | Nulo significa que no se aplicó ningún descuento |
| `fulfilled-by` | Rellenar | `"vendedor"` | Nulo significa que el vendedor gestionó el envío directamente |
| `Courier Status` (pedidos cancelados) | Rellenar según `Status` | `"cancelado"` | Lógica de negocio: un pedido cancelado no tiene courier activo |
| `Courier Status` (pedidos enviados) | Rellenar según `Status` | `"no registrado"` | El sistema no capturó el estado del courier en esos registros |
| `currency` (cancelados) | Rellenar según `Status` | `"cancelado"` | No aplica moneda a pedidos que no se completaron |
| `currency` (enviados / pendientes) | Rellenar según `Status` | `"inr"` | Única moneda del dataset (rupia india) |
| `Amount` | **Mediana por categoría** | Variable según categoría | La mediana es robusta a valores extremos; la categoría provee el contexto de precio correcto |

**Resultado: 0 valores nulos en todas las columnas relevantes.**
El dataset conserva **128.942 filas** — se eliminaron únicamente las 33 filas sin datos de ubicación.

## 4. Detección y Tratamiento de Outliers en `Amount`

Los valores extremadamente altos en el monto de venta pueden **sesgar métricas clave** como el LTV promedio, la rentabilidad por categoría y los clusters de segmentación de clientes.

Usamos el **método del Rango Intercuartil (IQR)**, el estándar estadístico para identificar outliers de forma robusta e independiente de la distribución de los datos:

**Fórmula:** `Límite superior = Q3 + 1.5 × IQR`

**Criterio de decisión:**
- Outliers **< 2%** del total → se **eliminan** (probablemente errores de carga o registros atípicos)
- Outliers **≥ 2%** del total → se **capean** al límite superior (son demasiados para descartar; preferimos acotar su efecto sin perder información)

In [13]:
# 4. Calcular cuartiles
'''
Los cuartiles son tres valores (Q1, Q2, Q3) que dividen un conjunto de datos ordenados en cuatro partes iguales, representando 
el 25%, 50% y 75% de la muestra. Son medidas de posición que facilitan el análisis de la distribución de los datos.

Primer Cuartil (Q1): El 25% de los datos son menores o iguales a este valor.
Segundo Cuartil (Q2): La mediana. El 50% de los datos son menores o iguales a este valor.
Tercer Cuartil (Q3): El 75% de los datos son menores o iguales a este valor.
'''

Q1 = df_clean['Amount'].quantile(0.25) # Calcular primer cuartil (Q1)
Q3 = df_clean['Amount'].quantile(0.75) # El tercer cuartil (Q3)
IQR = Q3 - Q1 # Muestra dónde se concentra el 50% central de los datos. Es el "corazón" de los datos, donde se concentra la mayoría normal.

'''
El IQR mide la dispersión de la mitad central de los datos y se utiliza para identificar outliers(valores atípicos).

outliers: observaciones numéricas que se alejan significativamente del resto de los datos en una muestra. Son puntos 
anómalos, inusualmente altos o bajos, que no siguen el patrón general y pueden sesgar el análisis estadístico, especialmente 
la media y la desviación estándar.

En concluisión, es una forma de discriminar los extremos y centrarnos en el rango donde se encuentra la mayoría de los datos normales.
'''

# Calcular límite superior (solo nos interesan valores altos)
limite_superior = Q3 + 1.5 * IQR
# El limite inferior sería Q1 - 1.5 *IQR (pero no nos interesa)
# 1.5 * IQR regla estándar para identificar valores atípicos (outliers) en un conjunto de datos.
# Si un valor es mayor que Q3 + 1.5 * IQR o menor que Q1 - 1.5 * IQR, se considera un outlier.

# Identificar outliers
# Creamos una máscara booleana donde True indica que la fila es un outlier (Amount > limite_superior) y False indica que no lo es.
outliers_mask = df_clean['Amount'] > limite_superior # Solo nos interesan los outliers altos, por eso comparamos con el límite superior.

# Contar cuántos outliers hay y qué porcentaje representan respecto al total de filas.
cantidad_outliers = outliers_mask.sum()
total_filas = len(df_clean)
porcentaje_outliers = (cantidad_outliers / total_filas) * 100

# Mostrar resultados
print("Detección de Outliers en 'Amount':\n")
print(f"Q1 (primer cuartil): {Q1:.2f}")
print(f"Q3 (tercer cuartil): {Q3:.2f}")
print(f"IQR (rango intercuartil): {IQR:.2f}")
print(f"Límite superior: {limite_superior:.2f}")
print(f"\nCantidad de outliers: {cantidad_outliers}")
print(f"Porcentaje de outliers: {porcentaje_outliers:.2f}%")

# Decisión según el porcentaje
if porcentaje_outliers < 2:
    print(f"\nDecisión: {porcentaje_outliers:.2f}% es menor al 2%. Se ELIMINAN los outliers.")
    df_clean = df_clean[~outliers_mask]  # Si dejamos [outliers_mask] dejamos la mascara de verdadero/falso como esta, al usar [~outliers_mask] invertimos la mascara y conservamos solo filas normales.
else:
    print(f"\nDecisión: {porcentaje_outliers:.2f}% es mayor o igual al 2%. Se CAPEAN los outliers.")
    df_clean.loc[outliers_mask, 'Amount'] = limite_superior  # Reducir al límite

# Verificar que ya no hay outliers
outliers_restantes = (df_clean['Amount'] > limite_superior).sum()
print(f"\nVerificación - Outliers restantes: {outliers_restantes}")
print(f"Nuevo tamaño del dataset: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas")

Detección de Outliers en 'Amount':

Q1 (primer cuartil): 437.14
Q3 (tercer cuartil): 788.00
IQR (rango intercuartil): 350.86
Límite superior: 1314.29

Cantidad de outliers: 3148
Porcentaje de outliers: 2.44%

Decisión: 2.44% es mayor o igual al 2%. Se CAPEAN los outliers.

Verificación - Outliers restantes: 0
Nuevo tamaño del dataset: 128942 filas y 22 columnas


## 5. Normalización de Texto

Las columnas de texto pueden contener el mismo valor escrito de formas distintas: `"BENGALURU"`, `"Bengaluru"` y `"bengaluru "` serían tratados por pandas como **tres ciudades diferentes**, generando agrupaciones incorrectas en cualquier análisis geográfico o de segmentación.

Aplicamos dos transformaciones a todas las columnas de tipo texto:

1. **`.str.strip()`** → elimina espacios en blanco al inicio y al final de cada valor
2. **`.str.lower()`** → convierte todo el texto a minúsculas

El impacto se mide comparando la cantidad de valores únicos antes y después de la normalización.

In [14]:
# Seleccionar automáticamente todas las columnas de tipo texto
columnas_texto = df_clean.select_dtypes(include=['object', 'string']).columns.tolist()

# Normalización de textos: minúsculas y sin espacios
print("Columnas a normalizar:", columnas_texto)

# Aplicar limpieza de texto: minúsculas y sin espacios
for col in columnas_texto:

    # Verificar que la columna existe en el dataset limpio y que es de tipo texto (object) antes de aplicar la normalización.
    if col in df_clean.columns and df_clean[col].dtype in ['object', 'string']:

        # Contar valores únicos antes de normalizar para comparar después.
        original_unicos = df_clean[col].nunique() 

        # El método .str.strip() elimina espacios en blanco al inicio y al final de cada valor, mientras que .str.lower() convierte todo el texto a minúsculas. 
        # Esto ayuda a unificar los valores y reducir la cantidad de categorías únicas causadas por diferencias de formato.
        df_clean[col] = df_clean[col].str.strip().str.lower() 

        # Contar valores únicos después de normalizar para ver el impacto de la limpieza.
        nuevos_unicos = df_clean[col].nunique()

        print(f"\n'{col}': {original_unicos} valores únicos → {nuevos_unicos} después de normalizar")

print("\nNormalización de textos completada.")

Columnas a normalizar: ['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'currency', 'ship-city', 'ship-state', 'ship-country', 'promotion-ids', 'fulfilled-by']

'Order ID': 120350 valores únicos → 120350 después de normalizar

'Date': 91 valores únicos → 91 después de normalizar

'Status': 13 valores únicos → 13 después de normalizar

'Fulfilment': 2 valores únicos → 2 después de normalizar

'Sales Channel ': 2 valores únicos → 2 después de normalizar

'ship-service-level': 2 valores únicos → 2 después de normalizar

'Style': 1377 valores únicos → 1377 después de normalizar

'SKU': 7195 valores únicos → 7195 después de normalizar

'Category': 9 valores únicos → 9 después de normalizar

'Size': 11 valores únicos → 11 después de normalizar

'ASIN': 7190 valores únicos → 7190 después de normalizar

'Courier Status': 5 valores únicos → 5 después de normalizar

'currency': 2 valores únicos → 2 

### Resultados de la Normalización

La mayoría de las columnas ya estaban bien formateadas. Las dos con mayor impacto fueron:

| Columna | Antes | Después | Reducción |
|---------|-------|---------|-----------|
| `ship-city` | 8.955 valores únicos | 7.297 | **−1.658 ciudades duplicadas eliminadas** |
| `ship-state` | 69 valores únicos | 47 | **−22 estados duplicados eliminados** |

> Estas reducciones **no representan pérdida de información**: eran duplicados causados por diferencias de capitalización (`"MUMBAI"` vs `"Mumbai"` vs `"mumbai"`). Al unificarlos, los grupos geográficos son más precisos y confiables para el análisis de distribución regional.

## 6. Conversión de Tipos de Datos

Dos columnas requieren un cambio de tipo para poder ser usadas correctamente en los análisis posteriores:

- **`Date`** está almacenada como texto (`str`). Para calcular días entre eventos, extraer el mes o analizar tendencias temporales, necesita ser de tipo `datetime`. El formato en el CSV es `MM-DD-YY` (mes-día-año anglosajón).
- **`B2B`** ya es booleano — se verifica por consistencia antes de continuar.

In [15]:
# 6. Convertir 'Date' a datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%m-%d-%y', errors='coerce')

# Verificar cuántas fechas no se pudieron convertir
# Contamos las fechas nulas porque se dejan como NaT (Not a Time) cuando no se pueden convertir correctamente.
fechas_nulas = df_clean['Date'].isna().sum() 

if fechas_nulas > 0:
    print(f"Atención: {fechas_nulas} fechas no se pudieron convertir (se dejaron como NaT)")
else:
    print("'Date' convertida correctamente")

# Convertir 'B2B' a booleano (ya debería ser bool, pero validamos)
if df_clean['B2B'].dtype != 'bool':
    df_clean['B2B'] = df_clean['B2B'].astype(bool)
    print("'B2B' convertida a booleano")
else:
    print("'B2B' ya era booleano")

# Verificar tipos finales
print(df_clean.dtypes)

'Date' convertida correctamente
'B2B' ya era booleano
Order ID                         str
Date                  datetime64[us]
Status                           str
Fulfilment                       str
Sales Channel                    str
ship-service-level               str
Style                            str
SKU                              str
Category                         str
Size                             str
ASIN                             str
Courier Status                   str
Qty                            int64
currency                         str
Amount                       float64
ship-city                        str
ship-state                       str
ship-postal-code             float64
ship-country                     str
promotion-ids                    str
B2B                             bool
fulfilled-by                     str
dtype: object


## 7. Feature Engineering — Creación de Variables Derivadas

El dataset crudo contiene datos transaccionales, pero **no las métricas necesarias para responder las preguntas de negocio**. En esta etapa creamos nuevas columnas que enriquecen el análisis en tres dimensiones:

| Variable | Tipo | Descripción | Pregunta de negocio |
|----------|------|-------------|---------------------|
| `Mes` | Temporalidad | Número de mes (1–12) | P3 — Estacionalidad |
| `Dia_Semana` | Temporalidad | Día de la semana (0=lun, 6=dom) | P3 — Patrones semanales |
| `Es_Finde` | Temporalidad | `True` si sábado o domingo | P3 — Comportamiento fin de semana |
| `Estacion` | Temporalidad | Estación del año según mes | P3 — Volatilidad estacional |
| `Canal_Adquisicion` | Canal | Amazon FBA vs. Merchant | P1 — Cohortes y LTV |
| `Cohorte_Mes` | Canal | Año-mes de la orden (ej. `"2022-04"`) | P1 — Análisis de retención |
| `Volatilidad_Demanda` | Producto | CV del Qty diario por SKU | P3 — Política de inventario |
| `Indice_Constancia` | Producto | Desvío estándar del gap entre ventas del SKU | P2 — Predictor de abandono |

> **Nota sobre RFM:** Las variables `Recencia`, `Frecuencia` y `Valor_Monetario` son métricas por cliente, no por transacción. Al no existir un identificador de cliente en este dataset, se calculan en `03_RFM_Clustering.ipynb` mediante una estrategia de agrupación definida allí.

### 7.1 Variables de temporalidad

In [16]:
# 7. Crear variables a partir de la fecha
df_clean['Mes'] = df_clean['Date'].dt.month # De la columna 'Date' se extrae el número de mes (1 a 12) y se guarda en la nueva columna 'Mes'.
df_clean['Dia_Semana'] = df_clean['Date'].dt.dayofweek # De la columna 'Date' se extrae el día de la semana (0=lunes a 6=domingo) y se guarda en la nueva columna 'Dia_Semana'.
df_clean['Es_Finde'] = (df_clean['Dia_Semana'] >= 5).astype(bool) # De la columna 'Dia_Semana' se extrae True si es sábado o domingo (5 o 6) y se guarda en la nueva columna 'Es_Finde'.

print("Variables creadas:")
print(f"   - 'Mes': número de mes (1 a 12)")
print(f"   - 'Dia_Semana': día de la semana (0=lunes a 6=domingo)")
print(f"   - 'Es_Finde': True si es sábado o domingo, False en caso contrario")

# Mostrar primeras filas para verificar
print("\nPrimeras 5 filas con las nuevas variables:")
print(df_clean[['Date', 'Mes', 'Dia_Semana', 'Es_Finde']].head())

Variables creadas:
   - 'Mes': número de mes (1 a 12)
   - 'Dia_Semana': día de la semana (0=lunes a 6=domingo)
   - 'Es_Finde': True si es sábado o domingo, False en caso contrario

Primeras 5 filas con las nuevas variables:
        Date  Mes  Dia_Semana  Es_Finde
0 2022-04-30    4           5      True
1 2022-04-30    4           5      True
2 2022-04-30    4           5      True
3 2022-04-30    4           5      True
4 2022-04-30    4           5      True


### 7.2 Variables de Canal, Estación y Cohorte

- **`Estacion`**: clasifica cada orden según la estación del año, base para el análisis de demanda estacional.
- **`Canal_Adquisicion`**: diferencia si el vendedor usó la logística de Amazon (`amazon` → `"Amazon FBA"`) o gestionó el envío de forma independiente (`merchant` → `"Merchant"`). Esta distinción es el proxy para comparar canales en el análisis de cohortes.
- **`Cohorte_Mes`**: registra el año-mes de cada orden en formato `"YYYY-MM"`, que actuará como identificador de cohorte mensual en el análisis de retención y LTV.

In [17]:
# Estación del año (meses según calendario meteorológico del hemisferio norte)
estaciones = {
    12: 'Invierno', 1: 'Invierno',  2: 'Invierno',
     3: 'Primavera', 4: 'Primavera', 5: 'Primavera',
     6: 'Verano',    7: 'Verano',    8: 'Verano',
     9: 'Otoño',    10: 'Otoño',    11: 'Otoño'
}
df_clean['Estacion'] = df_clean['Mes'].map(estaciones)

# Canal de Adquisición: Amazon FBA (logística delegada) vs. Merchant (logística propia)
df_clean['Canal_Adquisicion'] = df_clean['Fulfilment'].map({
    'amazon':   'Amazon FBA',
    'merchant': 'Merchant'
})

# Cohorte Mensual: año-mes de la orden en formato "YYYY-MM"
df_clean['Cohorte_Mes'] = df_clean['Date'].dt.to_period('M').astype(str)

print("Variables creadas: Estacion, Canal_Adquisicion, Cohorte_Mes\n")
print(f"Distribución Canal_Adquisicion:\n{df_clean['Canal_Adquisicion'].value_counts()}\n")
print(f"Distribución Estacion:\n{df_clean['Estacion'].value_counts()}\n")
print(f"Cohortes disponibles: {sorted(df_clean['Cohorte_Mes'].unique())}")

Variables creadas: Estacion, Canal_Adquisicion, Cohorte_Mes

Distribución Canal_Adquisicion:
Canal_Adquisicion
Amazon FBA    89678
Merchant      39264
Name: count, dtype: int64

Distribución Estacion:
Estacion
Primavera    91253
Verano       37689
Name: count, dtype: int64

Cohortes disponibles: ['2022-03', '2022-04', '2022-05', '2022-06']


### 7.3 Variables de Comportamiento de Producto

Calculamos dos métricas a nivel **SKU** que caracterizan el patrón de demanda de cada producto:

- **`Volatilidad_Demanda`** — Coeficiente de variación de las unidades vendidas por día: `CV = std / media`. Un valor alto indica demanda errática (difícil de anticipar); un valor bajo indica demanda estable. Base para la política de stock diferenciada.

- **`Indice_Constancia`** — Desvío estándar en días entre ventas consecutivas del mismo SKU. A menor valor, más constante es el ritmo de venta. Un SKU con solo un día de venta recibe valor `0` (no hay variabilidad medible).

Ambas métricas se calculan en forma agregada y luego se fusionan de vuelta al dataset fila por fila mediante un `merge` sobre `SKU`.

In [18]:
# --- Volatilidad_Demanda: CV = std/media de Qty diario por SKU ---
ventas_diarias = (
    df_clean.groupby(['SKU', 'Date'])['Qty']
    .sum()
    .reset_index()
)

volatilidad = (
    ventas_diarias.groupby('SKU')['Qty']
    .agg(media_diaria='mean', std_diaria='std')
    .reset_index()
)

# SKUs con un solo día de venta → std=NaN → CV=0 (sin variabilidad medible)
volatilidad['Volatilidad_Demanda'] = (
    (volatilidad['std_diaria'] / volatilidad['media_diaria'])
    .fillna(0)
    .round(4)
)

df_clean = df_clean.merge(
    volatilidad[['SKU', 'Volatilidad_Demanda']],
    on='SKU', how='left'
)

# --- Indice_Constancia: std de días entre ventas consecutivas del mismo SKU ---
df_sku_dates = (
    df_clean[['SKU', 'Date']]
    .drop_duplicates()
    .sort_values(['SKU', 'Date'])
)

df_sku_dates['gap_dias'] = (
    df_sku_dates.groupby('SKU')['Date']
    .diff()
    .dt.days
)

indice = (
    df_sku_dates.groupby('SKU')['gap_dias']
    .std()
    .fillna(0)
    .round(2)
    .reset_index()
    .rename(columns={'gap_dias': 'Indice_Constancia'})
)

df_clean = df_clean.merge(indice, on='SKU', how='left')

# --- Verificación ---
print("Variables creadas: Volatilidad_Demanda, Indice_Constancia\n")
print(f"Volatilidad_Demanda — resumen estadístico:")
print(df_clean['Volatilidad_Demanda'].describe().round(4))

print(f"\nIndice_Constancia — resumen estadístico:")
print(df_clean['Indice_Constancia'].describe().round(2))

print(f"\nTop 5 SKUs más volátiles:")
print(
    volatilidad.nlargest(5, 'Volatilidad_Demanda')
    [['SKU', 'media_diaria', 'std_diaria', 'Volatilidad_Demanda']]
    .reset_index(drop=True)
)

Variables creadas: Volatilidad_Demanda, Indice_Constancia

Volatilidad_Demanda — resumen estadístico:
count    128942.0000
mean          0.5598
std           0.2445
min           0.0000
25%           0.4410
50%           0.5793
75%           0.7046
max           1.8386
Name: Volatilidad_Demanda, dtype: float64

Indice_Constancia — resumen estadístico:
count    128942.00
mean          4.04
std           4.49
min           0.00
25%           1.34
50%           2.62
75%           5.03
max          59.40
Name: Indice_Constancia, dtype: float64

Top 5 SKUs más volátiles:
                  SKU  media_diaria  std_diaria  Volatilidad_Demanda
0  jne2305-kr-533-xxl      2.250000    4.136863               1.8386
1        j0150-kr-xxl      1.875000    3.313932               1.7674
2       btm038-pp-xxl      0.333333    0.577350               1.7321
3       btm044-pp-xxl      0.333333    0.577350               1.7321
4       j0133-kr-xxxl      0.333333    0.577350               1.7321


## 8. Exportación y Estado Final del Dataset

El ETL y el Feature Engineering están completos. Exportamos el dataset enriquecido a CSV — este archivo es la **fuente de verdad** para todos los notebooks de análisis del proyecto.

In [19]:
df_clean.to_csv('df_clean.csv', index=False)

print("\n=== RESUMEN FINAL DEL HITO 2 ===")
print(f"Dataset original: 128,975 filas")
print(f"Dataset limpio: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")


=== RESUMEN FINAL DEL HITO 2 ===
Dataset original: 128,975 filas
Dataset limpio: 128,942 filas x 30 columnas


---

## Conclusiones del Hito 2

### ¿Qué se logró?

| Transformación | Detalle |
|----------------|---------|
| **Nulos eliminados** | 0 nulos en todas las columnas relevantes (tratamiento contextual, no mecánico) |
| **Outliers tratados** | 3.148 registros de `Amount` capeados al límite IQR (₹1.314,29) |
| **Textos normalizados** | −1.658 ciudades y −22 estados duplicados unificados |
| **Tipos corregidos** | `Date` → `datetime64`, `B2B` verificado como `bool` |
| **Variables de temporalidad** | `Mes`, `Dia_Semana`, `Es_Finde`, `Estacion` |
| **Variables de canal** | `Canal_Adquisicion` (Amazon FBA vs. Merchant), `Cohorte_Mes` |
| **Variables de producto** | `Volatilidad_Demanda` (CV diario por SKU), `Indice_Constancia` (std de gaps entre ventas) |

### ¿Qué sigue?

El dataset `df_clean.csv` está listo para los notebooks de análisis:

| Notebook | Variables clave que usa |
|----------|------------------------|
| `02_Cohortes_LTV.ipynb` | `Cohorte_Mes`, `Canal_Adquisicion`, `Date`, `Amount` |
| `03_RFM_Clustering.ipynb` | `Order ID`, `Date`, `Amount` + estrategia de ID de cliente |
| `04_Estacionalidad_Inventario.ipynb` | `Estacion`, `Dia_Semana`, `Es_Finde`, `Volatilidad_Demanda`, `Indice_Constancia` |

> Las variables `Recencia`, `Frecuencia` y `Valor_Monetario` (RFM) se calculan en `03_RFM_Clustering.ipynb` porque son métricas por cliente — requieren una estrategia de agrupación que se define en ese notebook.